In [1]:
import json
import os

In [5]:
os.makedirs("data/json_files" , exist_ok=True)

In [6]:
# Sample nested JSON data

json_data = {
    "company": "TechCorp",
    "employees": [
        {
            "id": 1,
            "name": "John Doe",
            "role": "Software Engineer",
            "skills": ["Python", "JavaScript", "React"],
            "projects": [
                {
                    "name": "RAG System",
                    "status": "In Progress"
                },
                {
                    "name": "Data Pipeline",
                    "status": "Completed"
                }
            ]
        },
        {
            "id": 2,
            "name": "Jane Smith",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {
                    "name": "Customer Segmentation",
                    "status": "Completed"
                },
                {
                    "name": "Fraud Detection",
                    "status": "In Progress"
                }
            ]
        }
    ],
    "departments": [
        {
            "name": "Engineering",
            "manager": "John Doe",
            "budget": 500000
        },
        {
            "name": "Data Science",
            "manager": "Jane Smith",
            "budget": 300000
        }
    ]
}

In [7]:
with open("data/json_files/company_data.json", 'w') as f:
    json.dump(json_data,f, indent=2)

In [8]:
# Sample JSON Lines data

jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99},
    {"timestamp": "2024-01-02", "event": "user_logout", "user_id": 123},
    {"timestamp": "2024-01-02", "event": "user_login", "user_id": 456},
    {"timestamp": "2024-01-02", "event": "page_view", "user_id": 456, "page": "/products"},
    {"timestamp": "2024-01-02", "event": "add_to_cart", "user_id": 456, "product_id": "P001"},
    {"timestamp": "2024-01-02", "event": "purchase", "user_id": 456, "amount": 149.99}
]

In [9]:
with open("data/json_files/events.jsonl", 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + "\n")

### Json Processing Stratergies

In [11]:
from langchain_community.document_loaders import JSONLoader
import json

## Method 1 : JsonLoader with jq_schema
print("JSONLoader - Extract specific fields")

# Extract Employee Information
employee_loader = JSONLoader(
    file_path='data/json_files/company_data.json',
    jq_schema = '.employees[]', # jq query to extract each employee
    text_content= False # get full JSON objects
)

employee_docs = employee_loader.load()
print(f"Loaded {len(employee_docs)} employee documents")
print(f"First Employee: {employee_docs[0].page_content[:200]} .....")

JSONLoader - Extract specific fields
Loaded 2 employee documents
First Employee: {"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status" .....


In [12]:
employee_docs

[Document(metadata={'source': '/home/jyoti-singh/Downloads/course-target/ragudemy/0_DataInjestParsing/data/json_files/company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "John Doe", "role": "Software Engineer", "skills": ["Python", "JavaScript", "React"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'),
 Document(metadata={'source': '/home/jyoti-singh/Downloads/course-target/ragudemy/0_DataInjestParsing/data/json_files/company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Jane Smith", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "Customer Segmentation", "status": "Completed"}, {"name": "Fraud Detection", "status": "In Progress"}]}')]

In [15]:
# Method 2 : Custo JSON processing for complex structures

from typing import List
from langchain_core.documents import Document

print("\n Custom JSON Processing")

def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    
    with open(filepath, 'r') as f:
        data = json.load(f)
        
    documents = []
    
    # Strategy 1: Create documents for each employee with full context
    
    for emp in data.get('employees', []):
        content = f"""
        Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}
        Projects: 
        """
        for proj in emp.get('projects', []):
            content += f"\n- {proj['name']} (status: {proj['status']})"
            
        doc = Document(
            page_content= content,
            metadata= {
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        documents.append(doc)
    return documents
    


 Custom JSON Processing


In [17]:
process_json_intelligently("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'John Doe', 'role': 'Software Engineer'}, page_content='\n        Employee Profile:\n        Name: John Doe\n        Role: Software Engineer\n        Skills: Python, JavaScript, React\n        Projects: \n        \n- RAG System (status: In Progress)\n- Data Pipeline (status: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Jane Smith', 'role': 'Data Scientist'}, page_content='\n        Employee Profile:\n        Name: Jane Smith\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n        Projects: \n        \n- Customer Segmentation (status: Completed)\n- Fraud Detection (status: In Progress)')]